
# Variational Autoencoders (VAE) — Full Theory, Math & Backpropagation

This notebook includes:

• Deep intuition  
• Mathematical derivations  
• Reparameterization theory  
• KL divergence derivation  
• Full loss formulation  
• Explicit backpropagation math  
• Worked-out PyTorch gradient example  

The goal is full structural understanding — not memorization.



# 1️⃣ VAE Structure

Encoder:
x → μ(x), logσ²(x)

Latent sampling:
ε ~ N(0,1)  
z = μ + σ ε

Decoder:
x̂ = Decoder(z)

Loss:
L = Reconstruction(x, x̂) + KL(q || p)



# 2️⃣ KL Divergence (Gaussian vs Standard Normal)

For:

q(z|x) = N(μ, σ²)  
p(z) = N(0,1)

KL(q||p) = ½ Σ (μ² + σ² − logσ² − 1)

Interpretation:

• μ² term → penalizes drifting from zero  
• σ² term → penalizes large variance  
• −logσ² term → penalizes extremely small variance  



# 3️⃣ Reparameterization Trick (Core Math)

Original objective:

J(φ) = E_{z ~ q_φ(z|x)}[L(z)]

Reparameterized:

ε ~ N(0,1)  
z = μ_φ(x) + σ_φ(x) ε

Now:

J(φ) = E_{ε ~ N(0,1)}[L(μ + σ ε)]

Because ε does NOT depend on φ:

∇_φ J(φ) = E_ε [ ∇_φ L(μ + σ ε) ]

This allows gradients to pass through μ and σ.



# 4️⃣ Explicit Backpropagation Derivation

Let:

z = μ + σ ε  
L = L(z)

We compute:

∂L/∂φ = (∂L/∂z) (∂z/∂φ)

Now:

∂z/∂φ = ∂μ/∂φ + ε ∂σ/∂φ

Therefore:

∂L/∂φ =
(∂L/∂z)
(
∂μ/∂φ + ε ∂σ/∂φ
)

Key Insight:

ε is treated as a constant during that forward pass.
Gradients flow through μ and σ normally.



# 5️⃣ Worked Scalar Example

We verify the derivative numerically.


In [ ]:

import torch

torch.manual_seed(0)

# Define learnable parameters
mu = torch.tensor([1.0], requires_grad=True)
log_var = torch.tensor([0.0], requires_grad=True)

sigma = torch.exp(0.5 * log_var)

# Sample epsilon
epsilon = torch.randn(1)

# Reparameterization
z = mu + sigma * epsilon

# Simple loss
loss = z**2

# Backpropagation
loss.backward()

mu.grad, log_var.grad



# 6️⃣ Mini End-to-End VAE Forward + Backward Pass

We simulate one training step.


In [ ]:

import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Simple encoder and decoder
encoder = nn.Linear(4, 4)  # outputs μ and logσ² (2 each)
decoder = nn.Linear(2, 4)

x = torch.randn(4)

# Forward pass
encoded = encoder(x)
mu = encoded[:2]
log_var = encoded[2:]
sigma = torch.exp(0.5 * log_var)

epsilon = torch.randn(2)
z = mu + sigma * epsilon

x_hat = decoder(z)

# Reconstruction loss
recon_loss = F.mse_loss(x_hat, x)

# KL divergence
kl = 0.5 * torch.sum(mu**2 + sigma**2 - torch.log(sigma**2) - 1)

total_loss = recon_loss + kl

# Backpropagation
total_loss.backward()

total_loss



# 7️⃣ Final Structural Summary

Encoder outputs distribution parameters.

Reparameterization:
Isolates randomness into ε.

KL divergence:
Regularizes latent space geometry.

Reconstruction:
Encodes meaningful structure.

Backpropagation:
Flows through μ and σ via deterministic transformation.

This is the full mathematical and computational picture of VAE training.
